# Feedback Loop Trigger

## Objective
The loop fires when quality drifts, then self-corrects:

`trigger → migrate (response) → good enough? → deploy`

The live agent here uses Flash 2.5 (deployed by NB1). The trigger loop triggers migration to a newer model such as Flash 3.5 (evaluated locally as the candidate) and/or a hill climb exercise, such as GEPA prompt optimization. The performance ideally recovers above the threshold to be deployed downstream.

Note: the trigger here is the offline-eval path, run inline. In production the same loop fires from an online-monitor drift alert (console Preview) or a scheduled offline eval.

In [1]:
import os, json, importlib.util
import pandas as pd
from vertexai import Client, types

PROJECT = "sandbox-401718"           # @param {type:"string"}
AGENT = "brand-guidelines-checker"   # @param {type:"string"}
AGENT_DIR = f'../agents/{AGENT}'

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'true'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT

client = Client(project=PROJECT, location='global')
golden = json.load(open(f'{AGENT_DIR}/eval/golden.evalset.json'))
cases = golden['eval_cases']
BASELINE_MODEL = "gemini-2.5-flash"   # @param {type:"string"}
FRQ_THRESHOLD, SAFETY_THRESHOLD = 0.8, 1.0

def _custom_frq():
    """Custom Final Response Quality rubric (same as NB2)."""
    return types.LLMMetric(
        name='final_response_quality',
        prompt_template=(
            'You are grading a single-turn brand-guidelines checker.\n'
            'Marketing copy:\n{prompt}\n\n'
            "Checker's JSON verdict:\n{response}\n\n"
            'Top score only if every reported violation cites the correct brand rule id '
            'and the suggested_fix is concrete and would resolve it.\n'
            'Return ONLY JSON: {"score": <float 0-1>, "explanation": "<one sentence>"}'
        ),
        result_parsing_function=(
            'import json, re\n'
            'def parse_results(responses):\n'
            "    m = re.search(r'\\{.*\\}', responses[0], re.S)\n"
            '    d = json.loads(m.group(0)) if m else {}\n'
            "    return {'score': float(d.get('score', 0.0)), 'explanation': d.get('explanation', '')}\n"
        ),
    )

def eval_variant(model=None, instruction_file=None, deployed=None, runs=None):
    """Score the golden set, AVERAGED over `runs` passes to absorb judge/temperature
    noise (defaults to RUNS from the config cell). Pass deployed=<resource> to score the
    LIVE deployed agent, or model/instruction_file for a LOCAL candidate. -> [{id,frq,passed}]."""
    runs = runs if runs is not None else RUNS
    prompts = pd.DataFrame([
        {'prompt': c['input'],
         'session_inputs': types.evals.SessionInput(user_id='eval-user', app_name=AGENT)}
        for c in cases])
    if deployed:
        agent_arg, agent_info = deployed, None
    else:
        if model:
            os.environ['MODEL'] = model
        if instruction_file:
            os.environ['AGENT_INSTRUCTION_FILE'] = instruction_file
        else:
            os.environ.pop('AGENT_INSTRUCTION_FILE', None)
        spec = importlib.util.spec_from_file_location('variant', f'{AGENT_DIR}/app/agent.py')
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
        agent_arg = mod.root_agent
        agent_info = types.evals.AgentInfo.load_from_agent(agent=mod.root_agent)

    from collections import defaultdict
    acc = defaultdict(lambda: {'frq': [], 'safety': []})   # per-case scores across runs
    for _ in range(runs):
        inferred = client.evals.run_inference(agent=agent_arg, src=prompts)
        result = client.evals.evaluate(
            dataset=inferred,
            metrics=[_custom_frq(), types.RubricMetric.SAFETY, types.RubricMetric.GENERAL_QUALITY],
            agent_info=agent_info)
        for ecr in result.eval_case_results:
            mr = ecr.response_candidate_results[0].metric_results
            def sc(n): return float(mr[n].score) if mr.get(n) and mr[n].score is not None else 0.0
            cid = cases[ecr.eval_case_index]['id']
            acc[cid]['frq'].append(sc('final_response_quality'))
            acc[cid]['safety'].append(sc('safety_v1'))

    recs = []
    for c in cases:
        a = acc[c['id']]
        frq = sum(a['frq']) / len(a['frq']) if a['frq'] else 0.0
        safety = sum(a['safety']) / len(a['safety']) if a['safety'] else 0.0
        recs.append({'id': c['id'], 'frq': frq,
                     'passed': int(frq >= FRQ_THRESHOLD and safety >= SAFETY_THRESHOLD)})
    return recs


def bootstrap_ci(scores, resamples=2000, alpha=0.05, seed=0):
    import random
    rng = random.Random(seed); n = len(scores); means = []
    for _ in range(resamples):
        means.append(sum(scores[rng.randrange(n)] for _ in range(n)) / n)
    means.sort()
    return (sum(scores)/n, means[int(alpha/2*resamples)], means[int((1-alpha/2)*resamples)-1])

def threshold_meet(baseline, candidate, policy='match'):
    """Threshold check (same rule as eval_tool/compare.py): match live FRQ within the
    bootstrap noise band + zero regressions."""
    b = [r['frq'] for r in baseline]; c = [r['frq'] for r in candidate]
    bm, blo, bhi = bootstrap_ci(b); cm, clo, chi = bootstrap_ci(c)
    live_pass = {r['id']: r['passed'] for r in baseline}
    regressions = [r['id'] for r in candidate if r['passed'] == 0 and live_pass.get(r['id']) == 1]
    quality_ok = (clo > bm) if policy == 'beat' else (cm >= blo)
    ok = quality_ok and not regressions
    print(f'candidate FRQ={cm:.3f} CI[{clo:.3f},{chi:.3f}]  vs  live FRQ={bm:.3f} CI[{blo:.3f},{bhi:.3f}]')
    print(f'regressions: {regressions or "none"}  ->  THRESHOLD: {"PASS" if ok else "FAIL"}')
    return ok


# Quiet noise: google-genai's async client logs harmless teardown errors in notebooks
# (does not affect results). ADK's [EXPERIMENTAL] eval-feature warnings are left visible.
import warnings, logging
warnings.filterwarnings('ignore', message='.*was never awaited.*')
logging.getLogger('asyncio').setLevel(logging.CRITICAL)

/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


Set the drift bar and the candidate model below.

We also read the live agent's instruction here — not to simulate drift (the live agent is already drifted), but because the candidate (the newer model we're proposing) is evaluated locally before it's deployed, and for a fair "same agent, newer model" comparison it runs the same instruction as the live agent.

In [2]:
# --- loop config ---
# Two bars, two metrics, two stages (the 0.8s are coincidental, NOT the same knob):
DRIFT_THRESHOLD = 0.8                 # @param {type:"number"}   # TRIGGER on Final Response Quality (reference-free; prod has no answer key)
RUNS = 3                              # @param {type:"integer"}  # eval passes to average (denoise FRQ)
ACC_BAR        = 0.7                         # @param {type:"number"}   # GOOD-ENOUGH on verdict accuracy (offline, vs the golden answer key)
CANDIDATE_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
# Candidate inherits the live agent's instruction (read from the file NB1 deployed from).
LIVE_INSTRUCTION_FILE = os.path.abspath(f'{AGENT_DIR}/eval/weak_instruction.txt')
print('loop config ready · candidate inherits the live instruction:', LIVE_INSTRUCTION_FILE)

loop config ready · candidate inherits the live instruction: /home/jupyter/tmp/support/20240301_181108/jupyter/@ME/agent-ops-maturity/agent-ops-maturity-repotest/agents/brand-guidelines-checker/eval/weak_instruction.txt


## Feedback Loop

The sections below breaks down the following: Trigger → Model migration → Hill climb → Good enough? → Deploy. In the repo those same steps are a single program, `loop/run_ct_loop.py`, that runs them for real.

The driver: `python loop/run_ct_loop.py --agent-dir agents/<name>` is agent-agnostic — everything agent-specific (evalset, threshold policy, model) comes from the agent folder. It orchestrates the shared scripts:

| Step | Script | Does |
|------|--------|------|
| Hill climb | `loop/hillclimb_model.py` · `hillclimb_prompt.py` | produce a candidate (swap `model=` / optimize the prompt) |
| Evaluate | `eval_tool/run_eval.py` | score candidate + live on the golden set (`--engine local` or `managed`) |
| Good enough? | `eval_tool/compare.py` | threshold check: match live within the noise band, no regressions |
| Deploy | `deployment/deploy_agent.py` | update-in-place (new revision) into staging |

THE WIP Hands-off trigger *Automation* section is at the end.

## 1. Trigger — Detect Drift

Fetch the latest continuous-evaluation run logged from NB2 and read its Final Response Quality. If it's below the user-specified drift threshold, the loop fires.

In [3]:
import google.auth, google.auth.transport.requests, requests
from vertexai import Client as _Client
from google.genai import types as _gt

RUN_LOCATION = 'us-central1'   # NB2's runs live in the agent's region

# Find latest continuous-evaluation run (the SDK has no list, so use REST).
_creds, _ = google.auth.default()
_creds.refresh(google.auth.transport.requests.Request())
_url = (f"https://{RUN_LOCATION}-aiplatform.googleapis.com/v1beta1/"
        f"projects/{PROJECT}/locations/{RUN_LOCATION}/evaluationRuns?pageSize=100")
_runs = requests.get(_url, headers={'Authorization': f'Bearer {_creds.token}'}).json().get('evaluationRuns', [])
_mine = [r for r in _runs if r.get('displayName') == f'{AGENT}-offline-eval'
         and 'SUCCEEDED' in str(r.get('state', ''))]
assert _mine, f"No SUCCEEDED '{AGENT}-offline-eval' run found — run NB2 (Continuous Evaluation) first."
_latest = sorted(_mine, key=lambda r: r.get('createTime', ''))[-1]['name']

# Read that run's per-case Final Response Quality (native logged result — no re-eval).
_eval_client = _Client(project=PROJECT, location=RUN_LOCATION,
                       http_options=_gt.HttpOptions(api_version='v1beta1'))
run = _eval_client.evals.get_evaluation_run(name=_latest, include_evaluation_items=True)
baseline = []
for ecr in sorted(run.evaluation_item_results.eval_case_results, key=lambda e: e.eval_case_index):
    mr = ecr.response_candidate_results[0].metric_results
    def sc(n): return float(mr[n].score) if mr.get(n) and mr[n].score is not None else 0.0
    frq, safety = sc('final_response_quality'), sc('safety_v1')
    baseline.append({'id': cases[ecr.eval_case_index]['id'], 'frq': frq,
                     'passed': int(frq >= FRQ_THRESHOLD and safety >= SAFETY_THRESHOLD)})

baseline_frq = sum(r['frq'] for r in baseline) / len(baseline)
DRIFT = baseline_frq < DRIFT_THRESHOLD
print(f"NB2 run {_latest.split('/')[-1]} · live-agent FRQ = {baseline_frq:.3f}   drift_threshold = {DRIFT_THRESHOLD}")
print("DRIFT DETECTED -> firing the loop" if DRIFT
      else "no drift -> loop would not fire")

/var/tmp/ipykernel_717521/988409428.py:21: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_run module is experimental, and may change in future versions.
  run = _eval_client.evals.get_evaluation_run(name=_latest, include_evaluation_items=True)
/opt/conda/lib/python3.10/site-packages/vertexai/_genai/_evals_common.py:2759: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_set method is experimental, and may change in future versions.
  eval_set = evals_module.get_evaluation_set(
/opt/conda/lib/python3.10/site-packages/vertexai/_genai/_evals_common.py:2766: ExperimentalWarning: The Vertex SDK GenAI evals.get_evaluation_item method is experimental, and may change in future versions.
  evals_module.get_evaluation_item(name=item_name)


NB2 run 3666475454446960640 · live-agent FRQ = 0.620   drift_threshold = 0.8
DRIFT DETECTED -> firing the loop


## 2. Model Migration — to Flash 3.5

Turn the model iteration knob (2.5 → 3.5). The newer model handles the same instruction better; another technique is to hill-climb the prompt further (section below).

In [4]:
candidate = eval_variant(model=CANDIDATE_MODEL, instruction_file=LIVE_INSTRUCTION_FILE)
candidate_frq = sum(r['frq'] for r in candidate) / len(candidate)
print(f"candidate ({CANDIDATE_MODEL}): FRQ = {candidate_frq:.3f}")

Local Agent Run: 100%|██████████| 20/20 [00:11<00:00,  1.71it/s]
`kwargs` attribute in `evaluate` method is experimental and may change in future versions.
Local Agent Run: 100%|██████████| 20/20 [00:17<00:00,  1.13it/s]
`kwargs` attribute in `evaluate` method is experimental and may change in future versions.
Local Agent Run: 100%|██████████| 20/20 [00:24<00:00,  1.24s/it]
`kwargs` attribute in `evaluate` method is experimental and may change in future versions.
Computing Metrics for Evaluation Dataset: 100%|██████████| 60/60 [00:39<00:00,  1.52it/s]

candidate (gemini-3.5-flash): FRQ = 0.623


## 3. Hill Climb (optional) — Prompt Optimization with GEPA

Refine the migrated model with prompt optimization using ADK's documented GEPA optimizer (`GEPARootAgentPromptOptimizer` — reflective Genetic-Pareto search). It's scored on verdict accuracy vs the golden answer key — a deterministic signal, because on a small evalset an LLM rubric is too noisy to drive optimization. Starting from the live, rules-free instruction, GEPA reflects on the cases it gets wrong, discovers the brand rules, and rewrites the prompt. This hill-climbed instruction is what we deploy.

Note: Uses `loop/gepa_optimize.py`; needs google-adk ≥ 1.30 (this env is on 1.34.2).

In [5]:
import sys, os, importlib, nest_asyncio
import pandas as pd
from IPython.display import display
sys.path.insert(0, os.path.abspath('../loop'))
nest_asyncio.apply()   # GEPA's optimize() uses asyncio.run(); allow it inside Jupyter's loop
import gepa_optimize
importlib.reload(gepa_optimize)   # pick up latest gepa_optimize.py without a kernel restart

# GEPA hill-climb on the MIGRATED model, starting from the live (drifted) instruction.
# Scored on verdict accuracy vs the golden; writes results/optimized_instruction.txt.
# quiet=True hides GEPA's verbose per-iteration logs (set False to watch it reason).
gepa = gepa_optimize.run_gepa(agent_dir=AGENT_DIR, 
                              model=CANDIDATE_MODEL,
                              instruction_file=LIVE_INSTRUCTION_FILE, 
                              max_metric_calls=40, samples=2, 
                              # quiet=True
                             )

display(pd.DataFrame([{
    'verdict_acc_before': round(gepa['before_accuracy'], 3),
    'verdict_acc_after':  round(gepa['after_accuracy'], 3),
    'delta':              round(gepa['after_accuracy'] - gepa['before_accuracy'], 3),
    'prompt_rewritten':   gepa['changed'],
    'saved_to':           gepa['out_path'],
}]))

# Before vs after instruction (the 'after' is the deploy target in step 5).
# .get() + file fallback so a stale import can't break the cell.
before_instr = gepa.get('start_instruction') or open(LIVE_INSTRUCTION_FILE).read()
display(pd.DataFrame(
    {'instruction': [before_instr, gepa['instruction']]},
    index=['before (live/drifted)', 'after (GEPA-optimized)'],
).style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))

/opt/conda/lib/python3.10/site-packages/google/adk/evaluation/metric_evaluator_registry.py:112: UserWarning: [EXPERIMENTAL] MetricEvaluatorRegistry: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  metric_evaluator_registry = MetricEvaluatorRegistry()
/opt/conda/lib/python3.10/site-packages/google/adk/evaluation/local_eval_service.py:124: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  user_simulator_provider: UserSimulatorProvider = UserSimulatorProvider(),
/home/jupyter/tmp/support/20240301_181108/jupyter/@ME/agent-ops-maturity/agent-ops-maturity-repotest/loop/../eval_tool/run_eval.py:178: UserWarning: [EXPERIMENTAL] UserSimulatorProvider: This feature is experimental and may change or be removed in future versions without notice. It may introduce br

,verdict_acc_before,verdict_acc_after,delta,prompt_rewritten,saved_to
0,0.6,0.65,0.05,True,../agents/brand-guidelines-checker/results/opt...


## 4. Good Enough?

Check whether the results meet the threshold for downstream deployment.

In [6]:
recovered = gepa['after_accuracy'] >= ACC_BAR
print(f"hill-climbed verdict accuracy {gepa['after_accuracy']:.3f} vs bar {ACC_BAR}: "
      f"{'cleared' if recovered else 'still below'}")
print("GOOD ENOUGH -> deploy the GEPA-optimized candidate" if recovered
      else "NOT good enough -> raise GEPA's budget (max_metric_calls) and re-run")

hill-climbed verdict accuracy 0.650 vs bar 0.7: still below
NOT good enough -> raise GEPA's budget (max_metric_calls) and re-run


## 5. Deploy the Winner

Canary the hill-climbed candidate into eleevated environment e.g. staging env for the migrated model (Flash 3.5) with the optimized instruction.

Deploy here updates the agent in project. It uses the container/source deploy (same as NB1), so the new revision is a real `runtimeRevision`.

Documentation: [Manage deployed agents — update](https://docs.cloud.google.com/gemini-enterprise-agent-platform/scale/runtime/manage-deployed-agents#update)

![Agent revision](imgs/agent_revision.png)

Note: `deployment/deploy_agent.py` defaults to **`agents-cli deploy`** (source/container + identity); the inline cell here shows the equivalent SDK mechanics and bakes the GEPA-optimized instruction.


In [7]:
DEPLOY = True   # @param {type:"boolean"}

if DEPLOY:
    import os, shutil, yaml, vertexai
    from vertexai import agent_engines
    from google.genai import types as genai_types
    from vertexai._genai.types.common import AgentEngineConfig
    import vertexai._genai._agent_engines_utils as _aeu
    from google.adk.agents import Agent as _Agent

    REGION = 'us-central1'   # deploy is REGIONAL; model calls go global via env_vars below
    BUCKET_URI = f'gs://{PROJECT}-agent-staging'

    with open(f'{AGENT_DIR}/agent.yaml') as f:
        cfg = yaml.safe_load(f)
    display_name = cfg.get('display_name') or cfg.get('name', AGENT)
    description = cfg.get('description')

    # Promote exactly the hill-climbed candidate: migrated model + GEPA-optimized instruction,
    # deployed via the SAME source/container recipe as NB1 (so runtime revisions populate).
    CAND_INSTRUCTION = open(f'{AGENT_DIR}/results/optimized_instruction.txt').read().strip()
    BAKED_NAME = 'brand_guidelines_checker'   # valid identifier for the ADK Agent + AdkApp

    # Stage a dashless SOURCE package: app.py exposes a wrapped AdkApp ('app'); the optimized
    # instruction is baked into the image (instruction.txt). MODEL defaults to the candidate.
    BUILD_PKG = '_serving_build'
    shutil.rmtree(BUILD_PKG, ignore_errors=True); os.makedirs(BUILD_PKG)
    open(f'{BUILD_PKG}/app.py', 'w').write(
        'import os\n'
        'from google.adk.agents import Agent\n'
        'from vertexai import agent_engines\n'
        '_here = os.path.dirname(os.path.abspath(__file__))\n'
        f'MODEL = os.environ.get("MODEL", {CANDIDATE_MODEL!r})\n'
        'with open(os.path.join(_here, "instruction.txt")) as f:\n'
        '    INSTRUCTION = f.read().strip()\n'
        f'root_agent = Agent(name={BAKED_NAME!r}, model=MODEL, instruction=INSTRUCTION)\n'
        f'app = agent_engines.AdkApp(agent=root_agent, app_name={BAKED_NAME!r}, enable_tracing=True)\n'
    )
    open(f'{BUILD_PKG}/instruction.txt', 'w').write(CAND_INSTRUCTION)
    open(f'{BUILD_PKG}/requirements.txt', 'w').write(open(f'{AGENT_DIR}/requirements.txt').read())

    vertexai.init(project=PROJECT, location=REGION, staging_bucket=BUCKET_URI)
    # class_methods the source server registers (generated from a local AdkApp).
    _local_app = agent_engines.AdkApp(
        agent=_Agent(name=BAKED_NAME, model=CANDIDATE_MODEL, instruction=CAND_INSTRUCTION),
        app_name=BAKED_NAME)
    class_methods = [_aeu._to_dict(s) for s in _aeu._generate_class_methods_spec_or_raise(
        agent=_local_app, operations=_local_app.register_operations())]

    _env = {  # same as NB1: route model calls global + telemetry + prompt/response capture
        'GOOGLE_CLOUD_LOCATION': 'global',
        'GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY': 'true',
        'OTEL_SEMCONV_STABILITY_OPT_IN': 'gen_ai_latest_experimental',
        'OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT': 'EVENT_ONLY',
        'OTEL_INSTRUMENTATION_GENAI_UPLOAD_FORMAT': 'jsonl',
        'OTEL_INSTRUMENTATION_GENAI_COMPLETION_HOOK': 'upload',
        'OTEL_INSTRUMENTATION_GENAI_UPLOAD_BASE_PATH': f'{BUCKET_URI}/agent-payloads',
    }
    deploy_cfg = AgentEngineConfig(
        staging_bucket=BUCKET_URI, source_packages=[BUILD_PKG],
        entrypoint_module=f'{BUILD_PKG}.app', entrypoint_object='app',
        class_methods=class_methods, requirements_file=f'{BUILD_PKG}/requirements.txt',
        agent_framework='google-adk', env_vars=_env,
        display_name=display_name, description=description)

    client = vertexai.Client(project=PROJECT, location=REGION,
                             http_options=genai_types.HttpOptions(api_version='v1beta1'))
    # Upsert by display_name: update-in-place (new revision = lineage) else create.
    matches = [e for e in agent_engines.list()
               if (getattr(e, 'display_name', '') or '') == display_name]
    if matches:
        client.agent_engines.update(name=matches[0].resource_name, config=deploy_cfg)
        print('promoted (updated in place, new revision):', matches[0].resource_name)
    else:
        remote = client.agent_engines.create(config=deploy_cfg)
        name = (getattr(remote, 'api_resource', None) and remote.api_resource.name) or getattr(remote, 'name', None)
        print('deployed (new source engine):', name)
else:
    print('skipped deploy (DEPLOY is False) — set True to promote the winning candidate')


INFO:vertexai_genai.agentengines:Creating in-memory tarfile of source_packages
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=sandbox-401718&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%223538275147427348480%22.
INFO:vertexai_genai.agentengines:Agent Engine updated. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480')


promoted (updated in place, new revision): projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480


### Verify Deployed Revisions

The console's revision view is Preview and may lag or not show new revisions. Verify via the SDK (v1beta1). This lists the live agent's runtime revisions + state and an `ACTIVE` revision = a healthy deploy.

In [8]:
import yaml, pandas as pd
import vertexai
from vertexai import agent_engines
from google.genai import types as genai_types
from IPython.display import display

DEPLOY_REGION = 'us-central1'   # deploys are regional (eval client uses global; deploy/teardown regional)
vertexai.init(project=PROJECT, location=DEPLOY_REGION)
with open(f'{AGENT_DIR}/agent.yaml') as f:
    _name = yaml.safe_load(f).get('display_name') or AGENT
engine = next((e for e in agent_engines.list()
               if (getattr(e, 'display_name', '') or '') == _name), None)
assert engine is not None, f"no deployed engine named {_name!r} in {DEPLOY_REGION} (run the Deploy cell first)"
print('engine:', engine.resource_name)

# Runtime-revisions API (v1beta1). With the source/container deploy this NB now uses, each
# (re)deploy registers a revision, so this list grows by one per promotion. The list returns
# wrapper objects — the fields (name/state/create_time) live on r.api_resource.
client = vertexai.Client(project=PROJECT, location=DEPLOY_REGION,
                         http_options=genai_types.HttpOptions(api_version='v1beta1'))
revs = list(client.agent_engines.runtimes.revisions.list(name=engine.resource_name))
print(f'runtime revisions: {len(revs)}')
if revs:
    def _rev(r):
        o = getattr(r, 'api_resource', None) or r
        return {'revision': (getattr(o, 'name', '') or '').split('/')[-1],
                'state': str(getattr(o, 'state', '')),
                'created': str(getattr(o, 'create_time', ''))}
    display(pd.DataFrame([_rev(r) for r in revs]))

# THE REAL HEALTH CHECK: ask the deployed agent — works regardless of the revisions API.
sess = engine.create_session(user_id='verify')
chunks = []
for ev in engine.stream_query(user_id='verify', session_id=sess['id'],
                              message='MEGA SALE!!! our revolutionary blender is best-in-class'):
    for part in ev.get('content', {}).get('parts', []):
        if part.get('text'):
            chunks.append(part['text'])
resp = ''.join(chunks)
print('\nlive query ->', 'OK (agent is serving)' if resp.strip() else 'EMPTY (problem!)')
print(resp[:400])

engine: projects/757654702990/locations/us-central1/reasoningEngines/3538275147427348480
runtime revisions: 6


,revision,state,created
0,6,None,2026-06-05 04:51:44.219557+00:00
1,5,None,2026-06-05 04:31:25.930917+00:00
2,4,None,2026-06-04 22:46:17.646424+00:00
3,3,None,2026-06-04 22:28:24.944122+00:00
4,2,None,2026-06-03 21:00:59.276371+00:00
5,1,None,2026-06-03 20:31:36.040790+00:00



live query -> OK (agent is serving)
```json
{
  "verdict": "PASS",
  "violations": []
}
```


## (WIP - Optional Automation) Hands-off Trigger

*TODO: align with NB2 — leverage the Online Monitor to trigger the script.*

Everything above is human-on-demand. The production version runs the same loop with no human in the cycle:

```
Online Monitor (live traffic) → quality alert → Pub/Sub <agent>-loop-trigger → subscriber (Cloud Function) → run_ct_loop.py
```

- **Online Monitor** scores a sample of live traffic continuously and exports scores to Cloud Monitoring (`aiplatform.googleapis.com/online_evaluator/scores`). Console-only Preview today — set it up as described in NB2 (we don't re-implement it here).
- A **quality alert** on that metric fires on drift and publishes to the trigger topic via a Pub/Sub notification channel.
- The **subscriber** (`deployment/loop_subscriber/main.py`, a Cloud Function on `<agent>-loop-trigger`) runs `loop/run_ct_loop.py` unattended on each message — the same loop you stepped through above.

The scriptable pieces (trigger topic, notification channel, the subscriber function, and — as the GA alternative to the Preview Online Monitor — a log-based-metric drift alert) are provisioned by Terraform (`observability.tf` / `trigger.tf`). As with NB2's online monitor, this section describes the wiring rather than stitching Cloud Monitoring together by hand.

Note: this hands-off path auto-deploys to staging only; production promotion stays a manual approval gate.

Documentation: [Evaluate online](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/evaluate-online) · [Quality alerts](https://docs.cloud.google.com/gemini-enterprise-agent-platform/optimize/evaluation/quality-alerts)

```
drift alert → Pub/Sub → Cloud Function (loop_subscriber) → run_ct_loop.py
```

The driver runs on each drift message, no human in the loop. Auto-deploy is staging only; production promotion stays a manual gate.

Other orchestrators: the driver is a plain CLI, so the loop is orchestrator-agnostic. The default is event-driven (above); for scheduled runs, retries, or DAG dependencies, drive the same `run_ct_loop.py` from Airflow / Cloud Composer (a `BashOperator`) or Vertex AI Pipelines (a container component). Only what invokes the loop changes — the steps don't.